In [2]:
import pandas as pd
import numpy as np
import json
import joblib
import time

from pathlib import Path
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
RESULT_DIR = PROJECT_ROOT / "results" / "metrics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

X_train_text = train_df["clean_url"].fillna("").astype(str)
X_val_text = val_df["clean_url"].fillna("").astype(str)
X_test_text = test_df["clean_url"].fillna("").astype(str)

y_train = train_df["label_binary"].astype(int)
y_val = val_df["label_binary"].astype(int)
y_test = test_df["label_binary"].astype(int)

In [4]:
vectorizer = joblib.load(
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

X_train = vectorizer.transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

# XGBoost can accept sparse CSR matrices.
X_train = X_train.tocsr()
X_val = X_val.tocsr()
X_test = X_test.tocsr()

In [5]:
xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

training_time = time.perf_counter() - start_time

In [6]:
start_time = time.perf_counter()

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

inference_time = time.perf_counter() - start_time

metrics = {
    "model": "XGBoost",
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "training_seconds": round(training_time, 2),
    "inference_seconds": round(inference_time, 2),
    "test_samples": len(y_test)
}

joblib.dump(
    xgb_model,
    MODEL_DIR / "xgboost.joblib"
)

with open(
    RESULT_DIR / "xgboost_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)

In [7]:
print("=" * 60)
print("XGBOOST BASELINE RESULTS")
print("=" * 60)

print(f"Training samples : {len(y_train):,}")
print(f"Testing samples  : {len(y_test):,}")
print(f"TF-IDF features  : {X_train.shape[1]:,}")

print("-" * 60)

print(f"Accuracy         : {metrics['accuracy']:.4f}")
print(f"Precision        : {metrics['precision']:.4f}")
print(f"Recall           : {metrics['recall']:.4f}")
print(f"F1-score         : {metrics['f1_score']:.4f}")
print(f"ROC-AUC          : {metrics['roc_auc']:.4f}")

print("-" * 60)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("-" * 60)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malicious"],
    zero_division=0
))

print(f"Training time    : {training_time:.2f} seconds")
print(f"Inference time   : {inference_time:.2f} seconds")
print("Model saved      : models/xgboost.joblib")
print("=" * 60)

XGBOOST BASELINE RESULTS
Training samples : 448,779
Testing samples  : 96,167
TF-IDF features  : 30,000
------------------------------------------------------------
Accuracy         : 0.9746
Precision        : 0.9794
Recall           : 0.9434
F1-score         : 0.9610
ROC-AUC          : 0.9943
------------------------------------------------------------
Confusion Matrix:
[[63577   634]
 [ 1810 30146]]
------------------------------------------------------------
              precision    recall  f1-score   support

      Benign       0.97      0.99      0.98     64211
   Malicious       0.98      0.94      0.96     31956

    accuracy                           0.97     96167
   macro avg       0.98      0.97      0.97     96167
weighted avg       0.97      0.97      0.97     96167

Training time    : 5025.92 seconds
Inference time   : 1.48 seconds
Model saved      : models/xgboost.joblib
